# Using the Analytics Engine (AE) to reproduce annual consumption model
This notebook covers:
1. selecting data to work with
2. retrieving a dataset from the catalog
3. a simple plot to preview the data
4. how to export that data

To execute a given 'cell' of this notebook, place the cursor in the cell and press the 'play' icon, or simply press shift+enter together. Some cells will take longer to run, and you will see a [$\ast$] to the left of the cell while AE is still working.

## Step 0: Setup
This cell imports the python library [climakitae](https://github.com/cal-adapt/climakitae), our AE toolkit for climate data analysis, and any other specialized libraries needed for a given notebook.

In [ ]:
import numpy as np
from tqdm import tqdm # Progress bar 
import pyproj
import rioxarray
import pandas as pd
import xarray as xr
import hvplot.xarray
import hvplot.pandas
import panel as pn
pn.extension()

In [ ]:
import climakitae as ck

Additionally, get set up to make the computing go faster by executing the following cell. It will likely take several minutes to spin up! Learn more about dask and see some common [troubleshooting tips on our FAQ page](https://analytics.cal-adapt.org/docs/faq/).

In [ ]:
from dask.distributed import progress
from dask_gateway import GatewayCluster
cluster = GatewayCluster()
cluster.adapt(minimum = 0, maximum = 16)
client = cluster.get_client()
cluster

To use climakitae, load a new application:

In [ ]:
app = ck.Application()

## Step 1: Preview the data options
Now we can call 'select' to display an interface from which to select the data to examine. Execute the cell, and read on for more explanation.

Currently, you can select from [dynamically-downscaled](https://dept.atmos.ucla.edu/alexhall/downscaling-cmip6) data produced at hourly intervals. If you select 'daily' or 'monthly' for 'Timescale', you will receive an average of the hourly data. The spatial resolution options, on the other hand, are each the output of a different simulation, nesting to higher resolution over smaller areas.

Future projections are available for a [greenhouse gas emission scenario (Shared Socioeconomic Pathway, or SSP)](https://climatescenarios.org/primer/socioeconomic-development) through 2100 for SSP 3-7.0 for 5 General Circulation Models (GCMs).

At 45 and 9km, more GCMs are to come, and one GCM was also downscaled for a higher and lower SSP. (Later, statistical downscaling will also be available at 3km for more GCMs.)

"Historical Climate” includes data from 1980-2014 simulated from the same GCMs used to produce the SSPs. It will be automatically appended to a SSP time series when both are selected. Because this historical data is obtained through simulations, it represents average weather during the historical period and is not meant to capture historical timeseries as they occurred. 

“Historical Reconstruction” provides a reference downscaled [reanalysis](https://www.ecmwf.int/en/about/media-centre/focus/2020/fact-sheet-reanalysis) dataset based on atmospheric models fit to satellite and station observations, and as a result will reflect observed historical time-evolution of the weather.

To learn more about the data available on the Analytics Engine, [see our data catalog](https://analytics.cal-adapt.org/data/). 

In [ ]:
app.select()

Nothing is required to enter these selections, besides moving on to Step 2.

However, if you want to preview what has been selected, you can type "app.selections" alone in a new cell, and "app.location". These store your selections behind-the-scenes.

($+$ will create a new cell, following the currently selected) 

## Step 2: Retrieve the data
We can modify our data selections either using the `app.select` GUI, or in code, which we'll do below by modifying the `app.selections` object. 

### 2a) Chose your data and location subset

Here, we'll set the resolution, timescale, and scenarios.

In [ ]:
%%capture
app.selections.scenario_historical = ['Historical Climate']
app.selections.scenario_ssp = ['SSP 3-7.0 -- Business as Usual']
app.selections.area_average = False 
app.selections.timescale = 'hourly'
app.selections.resolution = '9 km'
app.selections.time_slice = (2005, 2025) 

Next, we'll set the location subset.

In [ ]:
%%capture
app.location.area_subset = 'states'
app.location.cached_area = 'CA' # Just get data for the state of California

### 2b) Retrieve data for each variable of interest
Set your desired variables (and corresponding units) by modifying the values in the dictionary `variable_units_dict` below. Any variables in this list must match a variable that is already available in our data catalog. **You can preview the variables and available units in the app.select GUI.** 

In [ ]:
# Dictionary in the format {variable : units}
variable_units_dict = { 
    "Air Temperature at 2m" : "degF", 
    "Relative Humidity": "[0 to 100]", 
    "Dew point temperature": "degF"
} 

To retrieve the data for all variables, we will simply loop through the variable list. Then, we'll compile the data for each variable into a simple xarray Dataset object, allowing us to easily subset and modify all the variables at once. 

In [ ]:
xr_list = [] # Empty list to store retrieved data for each variable 
for variable, unit in tqdm(variable_units_dict.items()): 
    
    # Reset variable and units in app.selections 
    app.selections.variable = variable
    app.selections.units = unit
    
    # Retrieve the data 
    xr_ds = app.retrieve() 
    xr_list.append(xr_ds)
    
# Merge data from all variables to form a single xr.Dataset object 
data = xr.merge(xr_list) 

### 2c) Preview the data 
You'll notice that each variable is a coordinate value in our output xarray dataset. 

In [ ]:
display(data) 

## Step 3: Get data from the closest grid cell to the Sacramento weather station. 

### 3a) Read in a csv file of the station coordinates 
We'll read it into a pandas DataFrame object.

In [ ]:
stations_df = pd.read_csv("CEC_Forecast_Weather Stations_California.csv", index_col = "STATION")
stations_df.head(5) # Display the first 5 rows 

### 3b) Grab the closest grid cell to the weather station
To demonstrate this process, we'll use the Sacreamento Executive Airport weather station.

In [ ]:
station_name = "SACRAMENTO EXECUTIVE AIRPORT"
one_station = stations_df.loc[station_name]

Next, we need to convert the lat/lon coordinate pair to the model's projection coordinates. We can easily do this using the python package pyproj by making a `Tranformer` object that will convert the coordinates for us.

In [ ]:
# Make Transformer object 
lat_lon_to_model_projection = pyproj.Transformer.from_crs(
    crs_from = "epsg:4326", # Lat/lon
    crs_to = data.rio.crs, # Model projection
    always_xy = True
)

# Convert station coordinates 
station_x, station_y = lat_lon_to_model_projection.transform(
    one_station.LON_X, # Station longitude 
    one_station.LAT_Y # Station latitude 
) 

Once we have the coordinates in the model projection, we can use [xarray's built in indexing function](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.sel.html), `.sel`, to easily grab the data from the grid cell that is closest to the weather station.

In [ ]:
data_closest_gridcell = data.sel(x = station_x, y = station_y, method = 'nearest')

### 3c) Load the data into memory 
This may take some time, because the data has to be loaded into memory and then subsetted to get the closest grid cell. All computations we've done before this step are actually computed in this step; before, we just see a preview of the data. 

In [ ]:
%%time
data_closest_gridcell = app.load(data_closest_gridcell)

Print the central coordinates of the nearest gridcell computed by xarray and compare to the actual station coordinates to observe how close they are.

In [ ]:
print("Weather station coords: (%.2f, %.2f)" % 
    (one_station.LAT_Y, one_station.LON_X) + "\n\
Nearest grid cell coords: (%.2f, %.2f)" % 
  (data_closest_gridcell.lat.item(), data_closest_gridcell.lon.item()))

## Step 4: Compute the median value of the grid cells in station's corresponding forecast zone
In the case of Sacramento, this is the Sacramento Municipal Utility District (SMUD)

### 4a) Read in the shapefiles of the demand forecast zones 
We'll use this to find the demand forecast zone that contains the weather station, then find the overlapping grid cells over which to compute the median value. The geometries of each demand forecast zone is available in our data catalog. You can grab the data as a pandas DataFrame object using the code provided below, or subset by forecast zones easily in `app.select` in the location subsetting tab. 

In [ ]:
%%time
from climakitae.selectors import Boundaries
dfzs_df = Boundaries()._ca_forecast_zones # Load geometries from catalog
dfzs_df.head(5) # Display the first 5 rows 

### 4b) Get the corresponding Demand Forecast Zone 
We'll use [geopanda's `.contains` function](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.contains.html) to find the demand forecast zone where the weather station is located, and print the result. 

In [ ]:
from shapely.geometry import Point

# Shapely Point object of the weather station 
shapely_Point = Point(one_station.LON_X, one_station.LAT_Y) 

# Get the name of the DFZ that contains that point 
forecast_zone = dfzs_df.where(dfzs_df.contains(shapely_Point)).dropna()
forecast_zone_name = forecast_zone.FZ_Name.item()
print("Weather station: {0} \nDemand forecast zone: {1}".format(station_name.lower().title(),forecast_zone_name))

### 4c) Crop the data to the forecast zone
We'll use rioxarray to clip the data to the geometry that defines the forecast zone. 

In [ ]:
data_dfz = data.rio.clip(
    geometries = [forecast_zone.geometry.item()], 
    crs = 4326, 
    drop = True # Drop cells outside the forecast zone 
)

## 4d) Visualize both the Demand Forecast Zone and the weather station on the same map 

For simplicity's sake, we'll show just one variable and only two weeks of data. In the outputted map, you can see that our data spans multiple dimensions and toggle between them using the dropdowns to the right of the map.   

In [ ]:
# Used to add weather station as star to map 
point_df = pd.DataFrame({
    "longitude (degrees_east)":[one_station.LON_X],
    "latitude (degrees_north)":[one_station.LAT_Y],
    "weather station": station_name
})

# Grab subset of data and load into memory 
to_plot = data_dfz["Air Temperature at 2m"].isel(time = np.arange(0,13))
to_plot = app.load(to_plot)

In [ ]:
app.view(to_plot) * point_df.hvplot.points(
    hover_cols = ["weather station"], 
    marker = "star", size = 300, color = "black"
)

### 4e) Agreggate values across grid cells in the forecast zone 
Chose your aggregation: median or mean. Either can be easily computed by xarray with just one line of code-- you can also compute min or mean the same way. You could also write your own code to compute a weighted mean. 

In [ ]:
# Chose your aggregation 
dfz_aggregation = "median"

In [ ]:
# Compute aggregation 
if dfz_aggregation == "median": 
    data_dfz_agreggated = data_dfz.median(dim = ["x","y"])
    
elif dfz_aggregation == "mean": 
    data_dfz_agreggated = data_dfz.mean(dim = ["x","y"])
    
# elif dfz_aggregation == "min": ... 
# elif dfz_aggregation == "weighted mean": ...

## Step 5: Select your input data 
Do you want to continue with this notebook using the single grid cell closest to the weather station, or the median value over the forecast zone? To set your input data, simply change the variable `input_datatype` to "aggregated dfz" or "closest grid cell".

In [ ]:
input_datatype = "aggregated dfz" 
#input_datatype = "closest grid cell"

In [ ]:
if input_datatype == "aggregated dfz":  
    data_to_use = data_dfz_agreggated
    num_grid_cells = data_dfz.x.size * data_dfz.y.size # Number of grid cells within the demand forecast region
    
elif input_datatype == "closest grid cell": 
    data_to_use = data_closest_gridcell
    num_grid_cells = 1

## Step 6: Compute daily multimodel statistics 
Here, we'll compute the mean, minimum, and maximum daily values for each temperature across all models. xarray makes this really simple: First, we'll resample the data to daily, then compute the mean over both the time and the simulation dimensions. 

In [ ]:
def resample_and_compute_statistics(data, resample_period="D"): 
    """Resample data and compute mean, min, and max across simulations and time 
    
    Args: 
        data (xr.Dataset): data to use         
        resample_period (str): "D" (daily) or "M" (monthly); default to daily 

    Returns: 
        data_mean, data_min, data_max (xr.Dataset)    
    """
    # Compute statistics 
    data_mean = data.resample(time = resample_period).mean(dim = ["simulation","time"])
    data_min = data.resample(time = resample_period).min(dim = ["simulation","time"]) 
    data_max = data.resample(time = resample_period).max(dim = ["simulation","time"]) 

    # Rename variables to reflect each aggregation 
    resample_str = {"D":"daily","M":"monthly"}[resample_period]
    data_mean = data_mean.rename({var:resample_str+" mean " + var for var in list(data.data_vars)})
    data_min = data_min.rename({var:resample_str+" min " + var for var in list(data.data_vars)})
    data_max = data_max.rename({var:resample_str+" max " + var for var in list(data.data_vars)})
    
    return (data_mean, data_min, data_max)

### 3a) Perform the computation 
We'll do this in one line of code using the function above.

In [ ]:
%%time
data_daily_mean, data_daily_min, data_daily_max = resample_and_compute_statistics(data_to_use)

### 3b) Output the results as a csv file
First, we'll convert the xarray Dataset object to a pandas DataFrame object. Then, we'll clean up the table and output the data to a csv file, making sure to give a descriptive name to the file to describe the computations and aggregations performed.  

In [ ]:
# Compile everything into the same xarray Dataset object
data_daily_ds = xr.merge([data_daily_mean, data_daily_min, data_daily_max])

In [ ]:
# %%time
# # Drop unwanted coordinates and convert to pandas DataFrame object 
# data_daily_df = data_daily_ds.isel(scenario = 0).drop(["Lambert_Conformal","scenario"]).to_dataframe()
# data_daily_df.head(5) # Show the first 5 rows 
# daily_csv_filename = (station_name+"_"+input_datatype).lower.replace(" ", "_")

## Step 7: Compute heating degree days and cooling degree days
Here, a heating degree day (HDD) is calculated by computing how many degrees Farenheit **colder** the daily temperature is from a standard temperature of 65 degrees Farenheit. A cooling degree day (CDD) is calulcated by computing how many degrees **warmer** the daily temperature is from the same standard temperature.

### 7a) Compute HDD and CDD 
HDD = 65 - temperature<br>
CDD = (-1)\*(65 - temperature)<br><br>
For HDD, we can just subtract the 2m temperature from 65 degrees Farenheight, then set any negative to 0. For CDD, we will do the same, but will then multiply by -1 to turn negative values to positive, then set negative values to 0. We need to multiply by -1 for CDD to avoid having all negative values; for example, on a day of 80F, CDD = 65 - 80 = -15, but the CDD value is +15. Multiplying -15 by -1 will give us the true value of 15. 

In [ ]:
def compute_hdd_cdd(t2, standard_temp = 65): 
    """Compute heating degree days (HDD) and cooling degree days (CDD) 
    
    Args: 
        t2 (xr.DataArray): Air temperature at 2m gridded data 
        standard_temp (int, optional): standard temperature in Fahrenheit (default to 65)
        
    Returns: 
        hdd, cdd (xr.DataArray) 
    """
    # Subtract t2 from the standard reference temperature 
    deg_less_than_standard = 65 - t2

    # Compute HDD: Find positive difference (i.e. days < 65 degF) 
    hdd = deg_less_than_standard.where(deg_less_than_standard > 0, 0) # Replace negative values with 0
    hdd.name = "Heating Degree Days" 

    # Compute CDD: Find negative difference (i.e. days > 65 degF)
    cdd = (-1)*deg_less_than_standard.where(deg_less_than_standard < 0, 0) # Replace positive values with 0
    cdd.name = "Cooling Degree Days" 
    
    return (hdd, cdd) 

In [ ]:
t2 = data_to_use["Air Temperature at 2m"]
hdd, cdd = compute_hdd_cdd(t2, standard_temp = 65)

### 7b) Aggregate annually to find HDD and CDD per year
First, we will group the data by year and compute a sum across space and time. Then, we will divide the annual aggregated data by the number of grid cells over which the sum was computed. 

In [ ]:
def compute_annual_aggreggate(data, name, num_grid_cells):  
    annual_ag = data.squeeze().groupby('time.year').sum(['time']) # Aggregate annually
    annual_ag = annual_ag/num_grid_cells # Divide by number of gridcells 
    annual_ag.name = name # Give new name to dataset
    return annual_ag

In [ ]:
hdd_annual = compute_annual_aggreggate(
    data = hdd, 
    name = "Annual Heating Degree Days (HDD)", 
    num_grid_cells = num_grid_cells
)
cdd_annual = compute_annual_aggreggate(
    data = cdd, 
    name = "Annual Cooling Degree Days (CDD)", 
    num_grid_cells = num_grid_cells
)

### 7c) Compute a trendline using the mean of all simulations
First, we'll compute the mean across all simulations. Then, we'll find the coefficients for a first degree (linear) polynomial using [numpy's `polyfit` function](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html). The returned coefficients (**m** and **b** in the code below) will allow us to compute the trendline using the linear polynomial y = mx + b, where **y** is the trendline and **x** is the years. 

In [ ]:
def trendline(data): 
    years = data.year # These are the x values 
    data_mean = data.mean(dim = "simulation") # Compute mean across all simulations 
    m, b = np.polyfit(
        x = years.values, 
        y = data_mean.values, 
        deg = 1
    ) 
    trendline = m*years + b # y = mx + b 
    trendline.name = "trendline" 
    return trendline

In [ ]:
hdd_trendline = trendline(hdd_annual) 
cdd_trendline = trendline(cdd_annual) 

### 7d) Visualize the results
Using the python package *hvplot*, we can easily make a line plot of the annual aggregated data. To do this, we'll plot the annual HDD, then add the trendline on top. The code to generate the plot is contained in a function `plot_data` defined below. 

In [ ]:
def plot_data(annual_data, trendline, title = "title"): 
    return annual_data.hvplot.line(
        x = "year", by = "simulation", 
        width = 800, height = 350,
        title = title,
        yformatter = '%.0f' # Remove scientific notation
    ) * trendline.hvplot.line(  # Add trendline
        x = "year", 
        color = "black", 
        line_dash = 'dashed', 
        label = "trendline"
    ) 

In [ ]:
plot_data(
    annual_data = hdd_annual, 
    trendline = hdd_trendline, 
    title = "Annual Aggregate Heating Degree Days"
)

In [ ]:
plot_data(
    annual_data = cdd_annual, 
    trendline = cdd_trendline, 
    title = "Annual Aggregate Cooling Degree Days"
)

### 7e) Output data as csv files
We'll drop all unneeded coordinates and convert our xarray Dataset to a pandas Dataframe, allowing us to easily output the final data product to a csv file. 

In [ ]:
# Pick a simulation 
simulation = "cnrm-esm2-1"

# Simplify data 
station_data_simple = data_to_use.isel(scenario = 0).sel(simulation = simulation).drop(
    ["simulation","Lambert_Conformal","scenario"] # Unwanted coordinates
)

# Convert to pandas dataframe 
station_data_df = station_data_simple.to_dataframe()
station_data_df.head()

In [ ]:
station_data_df.to_csv("my_data.csv")

## Step 8: Perform the same analyses for all weather stations
We'll loop through each weather station and perform the same analyses as above: (1) daily min/max/mean and (2) annual aggreggate HDD/CDD)

### 8a) Select your input data and aggregation type 
Do you want to continue with this notebook using the single grid cell closest to the weather station, or the median value over the forecast zone? To set your input data, simply change the variable `input_datatype` to "aggregated dfz" or "closest grid cell". 

Also, you'll need to chose the type of aggregation you want to perform within the DFZ. 

In [ ]:
# input_datatype = "closest grid cell" 
# input_datatype = "aggregated dfz" 
# dfz_aggregation = "median"

### 8b) Run the analysis 
Everything from the notebook has been condensed into several helper functions, which will be run together to perform the workflow for each weather station. 

In [ ]:
# def _setup_analysis():
#     """Retrieve data from catalog. Returns Dataset object. """
    
#     # Reset app.selections and app.location to desired values 
#     app.selections.scenario_historical = ['Historical Climate']
#     app.selections.scenario_ssp = ['SSP 3-7.0 -- Business as Usual']
#     app.selections.area_average = False 
#     app.selections.timescale = 'hourly'
#     app.selections.resolution = '9 km'
#     app.selections.time_slice = time_slice 
    
#     # Dictionary in the format {variable : units}
#     variable_units_dict = { 
#         "Air Temperature at 2m" : "degF", 
#         "Relative Humidity": "[0 to 100]", 
#         "Dew point temperature": "degF"
#     } 
    
#     # Retrieve catalog data for each variable 
#     xr_list = [] 
#     for variable, unit in tqdm(variable_units_dict.items()): 
#         app.selections.variable = variable
#         app.selections.units = unit
#         xr_ds = app.retrieve() 
#         xr_list.append(xr_ds)
#     data = xr.merge(xr_list) 
#     return data 

In [ ]:
# def _subset_data(data, stations_df, input_datatype = "aggregated dfz", dfz_aggregation = "median"): 
        
#     if input_datatype == "closest grid cell": 
            
#             lat_lon_to_model_projection = pyproj.Transformer.from_crs(
#                 crs_from = "epsg:4326", # Lat/lon
#                 crs_to = data.rio.crs, # Model projection
#                 always_xy = True
#             )
#             station_x, station_y = lat_lon_to_model_projection.transform(
#                 one_station.LON_X, # Station longitude 
#                 one_station.LAT_Y # Station latitude 
#             ) 
#             data_closest_gridcell = data.sel(x = station_x, y = station_y, method = 'nearest')
#             data_to_use = data_closest_gridcell
#             num_grid_cells = 1

#     if input_datatype == "aggregated dfz" : 

#         # Subset data to input DFZ 
#         dfzs_df = Boundaries()._ca_forecast_zones 
#         shapely_Point = Point(one_station.LON_X, one_station.LAT_Y) 
#         forecast_zone = dfzs_df.where(dfzs_df.contains(shapely_Point)).dropna()
#         data_dfz = data.rio.clip(
#             geometries = [forecast_zone.geometry.item()], 
#             crs = 4326, 
#             drop = True 
#         )

#         # Compute aggregation 
#         if dfz_aggregation == "median": 
#             data_dfz_agreggated = data_dfz.median(dim = ["x","y"])
#         elif dfz_aggregation == "mean": 
#             data_dfz_agreggated = data_dfz.mean(dim = ["x","y"])

#         data_to_use = data_dfz_agreggated
#         num_grid_cells = data_dfz.x.size * data_dfz.y.size # Number of grid cells within the demand forecast region

#     return (data_to_use, num_grid_cells) 

## Step 9: Close the compute cluster
Lastly, when you are done, close your cluster resources to free them up for the next time you work. 

In [ ]:
client.close()